In [ ]:
# %pip install numpy_financial
# %pip install pandas
# %pip install numpy
# %pip install langchain_google_genai

In [4]:
import os
import textwrap
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np

from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.getenv('GEMINI_API_KEY')

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))


In [5]:
def gemini_response(model, instruction, count):
    scores = []

    for i in range(count):
        try:
            result = model.invoke(instruction)
            scores.append(int(result.content))
        except:
            pass

    if len(scores) == 0:
        score = 0
    else:
        score = sum(scores) / len(scores)

    return score  # 分數

In [6]:
def calculate_irr(signal, threshold):
    # [日期, 評分, 價格]
    hold = 0  # 持有的股數
    balance_in = []  # 每月投入的資金
    balance_total = 0
    month = 1
    this_month_in = 0
    for item in signal:
        score = float(item[1])
        price = float(item[2])
        if score > threshold:
            balance_total += score * 1000  # 買入分數*1000等值的股票
            hold = hold + ((score * 1000) / price)  # 計算持有的股票數

            # 計算每月投入金額
            date_obj = datetime.strptime(item[0], '%Y%m%d')
            if month == date_obj.month:
                this_month_in += score * 1000
            elif month != date_obj.month:
                month = date_obj.month
                this_month_in += score * 1000
                balance_in.append(-this_month_in)
                this_month_in = 0

    balance_in.append(-this_month_in)
    last_price = float(signal[len(signal) - 1][2])
    
    balance_in.append(hold * last_price)
    # print(balance_in)
    irr = npf.irr(balance_in)

    if math.isnan(irr):
        irr = 0

    if balance_total > 0:
        bnh = last_price / float(signal[0][2])
    else:
        bnh = 0

    return round(irr, 5), round(bnh, 5)

In [24]:
def mutate_prompt(llm, p_mutate):
    prompt = '產生以下指令的1個變體，同時保留語意：'
    response = llm.invoke(prompt + p_mutate)
    return response.content

In [9]:
# main cell
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = secret.GEMINI_API_KEY

regenerate_count = 3
prompt_mutate_count = 100

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

stock_ids = ["1101", "2211", "2385", "2542", "2880", "2912", "3023", "3264", "5269", "8027"]
# stock_ids = ["2912", "3023", "3264", "5269", "8027"]

llm = ChatGoogleGenerativeAI(
    model="gemini-pro",
    temperature=1,
    safety_settings={
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    })

p_mutate = '深入的公司創新分析及其對預期市場影響的評估，表明股價持續上漲，反映了投資者對公司令人振奮的長期增長前景的樂觀情緒。'
irr_ = 0.56

for i in range(prompt_mutate_count):
    # 變異
    p_mutate_2 = mutate_prompt(llm, p_mutate)
    general_prompt = '請依據' + p_mutate_2 + '給這篇文章打一個分數，分數介於-100～100之間，-100是負向，100是最好，0則是無法判斷'

    irrs = []
    bnhs = []
    raw_data = []
    for stock_id in stock_ids:
        print(f'變異{i + 1}次\t執行{stock_id}')
        folder_path = "stock_news/" + stock_id + "/" + stock_id + "_news/"
        files = os.listdir(folder_path)
        sorted_files = sorted(files)

        # 載入買進價格
        df = pd.read_csv("price_history/" + stock_id + ".csv", dtype=str)
        price_np = df.to_numpy()
        prices = price_np.tolist()  # [時間,價格]

        signal_2 = []

        for price in prices:
            # 檢查日期是否在參考日期之後
            try:
                date_obj = datetime.strptime(price[0], '%Y%m%d')
            except:
                continue

            if not (start_time <= date_obj <= end_time):
                continue
            
            sig = 0
            for index, file_name in enumerate(sorted_files):
                if file_name == price[0]:
                    with open(folder_path + file_name, encoding='utf-8') as f:
                        news = f.read()
                        # print(news)
                        prompt = general_prompt + "\n" + news
                        # gemini的回應，回傳帶有買入訊號的list
                        sig = gemini_response(llm, prompt, regenerate_count)
                    break

            signal_2.append([price[0], sig, price[1]])

        signal_2_np = np.array(signal_2)
        # 回傳 IRR 以及總額投入
        irr, bnh = calculate_irr(signal_2_np, 0)
        irrs.append(irr)
        bnhs.append(bnh)
        raw_data.append(signal_2_np.tolist())
        # print("每月內部報酬率", irr)
        # print("總額投入報酬率", bnh)

    # 輸出結果
    write_folder = "evaluate_output/output"
    count = 1
    while True:
        if os.path.exists(write_folder + str(count)):
            count += 1
            continue
        else:
            with open(write_folder + str(count), "a", encoding="UTF-8") as f:
                content = (f'提示：{p_mutate_2}\n mutate from {p_mutate}\n'
                           f'開始日期 {start_time.strftime("%Y%m%d")},'
                           f'結束日期 {end_time.strftime("%Y%m%d")}\n')
                for j in range(len(stock_ids)):
                    content += (f'股票代號：{stock_ids[j]}\t'
                                f'每月內部報酬率:{irrs[j]}\t'
                                f'總額投入報酬率:{bnhs[j]}\n')
                content += f'變異{i + 1}次，平均內部報酬率:{sum(irrs) / len(irrs)}，平均總額投入報酬率:{sum(bnhs) / len(bnhs)}\n)'
                content += f'原始數據{raw_data}'
                f.write(content)
            print("寫入" + write_folder + str(count))
            break

    # 判斷是否更改變異
    if sum(irrs) / len(irrs) > irr_:
        print("更改變異提示為:", p_mutate_2)
        p_mutate = p_mutate_2
        irr_ = sum(irrs) / len(irrs)

# todo testing


變異1次	執行1101


In [15]:
#test cell
# with open("prompt template/temp3", "r") as f:
#     prompt_temp = f.read()

p1 = '請依據〔記者梁世煌／台北報導〕水泥股公司派近期護盤動作轉趨明顯，其中董座陳敏賢遭高捷案牽扯的東南水泥（1110）決定實施庫藏股1萬張，台泥（1101）則是透過轉投資恆強投資及富品投資持續回補自家股票。市場人士指出，外資前不久持續調節減碼水泥股，面臨明年董監改選的變數，公司派加快回補，一方面護盤、一方面為明年的董監選情佈局。〔記者梁世煌／台北報導〕水泥股公司派近期護盤動作轉趨明顯，其中董座陳敏賢遭高捷案牽扯的東南水泥（1110）決定實施庫藏股1萬張，台泥（1101）則是透過轉投資恆強投資及富品投資持續回補自家股票。市場人士指出，外資前不久持續調節減碼水泥股，面臨明年董監改選的變數，公司派加快回補，一方面護盤、一方面為明年的董監選情佈局。台泥回補 佈局董監改選台泥回補 佈局董監改選根據統計，由於明年台泥即將進行董監改選，今年下半年以來，台泥進場加碼動作頻頻，以10月為例，台泥轉投資恆強投資就買進台泥股票9177張，9月時，台泥轉投資富品投資亦買進台泥股票4242張，恆強投資亦加碼了5554張，而台泥副董事長張安平亦有小量回補，累計今年7月至今，富品、恆強加上辜成允、中橡等主要大股東的持股比重已達1成以上。請繼續往下閱讀...不過，台泥近期來遭外資持續減碼，外資持股比例較10月份下降8個百分點，目前僅有24.57%，由於台泥籌碼目前仍十分凌亂，分析師認為，台泥大舉回補自家股票有趁勢為明年董監改選佈局的用意。嘉泥投資台泥 力挺辜家嘉泥投資台泥 力挺辜家另外，嘉新水泥（1103）昨天公告子公司嘉新投資以每股20.49元的價格買入台泥股票1.47萬張，由於嘉新張家與台泥辜家關係匪淺，面對市場派全力搶攻，嘉新的動作被視為有力挺辜家的意義。東泥要買回1萬張東泥要買回1萬張另外，東泥日前亦公告實施庫藏股，買回自家股票1萬張，由於東泥董座陳敏賢目前飽受高捷案困擾，對東泥的股價造成負面影響，因此，東泥決定實施庫藏股護盤意味十足，而東泥股價昨天亦以5.84元收紅。一手掌握經濟脈動給這篇文章打一個分數，分數介於-100～100之間，-100是負向，100是最好，0則是無法判斷'
# p2 = "請用一句話產生一個選股條件"
llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)

# with open("stock_news/test1", "r") as f:
#     news = f.read()

# prompt = prompt_temp + "\n" + news

result = llm.invoke(p1)

print(f'prompt: {p1}')
print("__")
print(f'response: {result.content}')
print("_"*50)

prompt: 請依據〔記者梁世煌／台北報導〕水泥股公司派近期護盤動作轉趨明顯，其中董座陳敏賢遭高捷案牽扯的東南水泥（1110）決定實施庫藏股1萬張，台泥（1101）則是透過轉投資恆強投資及富品投資持續回補自家股票。市場人士指出，外資前不久持續調節減碼水泥股，面臨明年董監改選的變數，公司派加快回補，一方面護盤、一方面為明年的董監選情佈局。〔記者梁世煌／台北報導〕水泥股公司派近期護盤動作轉趨明顯，其中董座陳敏賢遭高捷案牽扯的東南水泥（1110）決定實施庫藏股1萬張，台泥（1101）則是透過轉投資恆強投資及富品投資持續回補自家股票。市場人士指出，外資前不久持續調節減碼水泥股，面臨明年董監改選的變數，公司派加快回補，一方面護盤、一方面為明年的董監選情佈局。台泥回補 佈局董監改選台泥回補 佈局董監改選根據統計，由於明年台泥即將進行董監改選，今年下半年以來，台泥進場加碼動作頻頻，以10月為例，台泥轉投資恆強投資就買進台泥股票9177張，9月時，台泥轉投資富品投資亦買進台泥股票4242張，恆強投資亦加碼了5554張，而台泥副董事長張安平亦有小量回補，累計今年7月至今，富品、恆強加上辜成允、中橡等主要大股東的持股比重已達1成以上。請繼續往下閱讀...不過，台泥近期來遭外資持續減碼，外資持股比例較10月份下降8個百分點，目前僅有24.57%，由於台泥籌碼目前仍十分凌亂，分析師認為，台泥大舉回補自家股票有趁勢為明年董監改選佈局的用意。嘉泥投資台泥 力挺辜家嘉泥投資台泥 力挺辜家另外，嘉新水泥（1103）昨天公告子公司嘉新投資以每股20.49元的價格買入台泥股票1.47萬張，由於嘉新張家與台泥辜家關係匪淺，面對市場派全力搶攻，嘉新的動作被視為有力挺辜家的意義。東泥要買回1萬張東泥要買回1萬張另外，東泥日前亦公告實施庫藏股，買回自家股票1萬張，由於東泥董座陳敏賢目前飽受高捷案困擾，對東泥的股價造成負面影響，因此，東泥決定實施庫藏股護盤意味十足，而東泥股價昨天亦以5.84元收紅。一手掌握經濟脈動給這篇文章打一個分數，分數介於-100～100之間，-100是負向，100是最好，0則是無法判斷
__
response: 70
__________________________________________________


In [23]:
p = "深入的公司創新分析及其對預期市場影響的評估，表明股價持續上漲，反映了投資者對公司令人振奮的長期增長前景的樂觀情緒。"
p1 = mutate_prompt(llm, p)

# llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)
# result = llm.invoke(p1)

print(f'prompt: {p1}')
print("__")
print(f'response: {result.content}')
print("_"*50)

prompt: 全面深入剖析企業的創新舉措，評估其預期市場影響，顯著股價階梯式上升，反映出投資人對其未來亮眼增長前景的熱切期待。
__
response: **公司的創新策略**

公司採取全面性的創新策略，以保持其在業界的領先地位。關鍵要素包括：

* **研發投資：**公司持續投入大量資金於研發活動，專注於開發新產品、改良現有技術和探索新興領域。
* **開放創新：**公司積極與外部合作夥伴、學術機構和初創企業合作，激發新的想法和加速創新進度。
* **客戶導向：**公司密切關注顧客反饋，並將客戶需求整合到其創新過程中。通過徹底了解客戶痛點和預期，公司可創造符合市場需求的解決方案。

**對目標市場的預期影響**

公司的創新策略預計將對其目標市場產生重大影響：

* **提升客戶價值：**新產品和服務旨在解決未滿足的需求，提供更好的價值和改善客戶體驗。
* **擴展產品組合：**不斷創新使公司得以擴展其產品組合，進入新的市場領域，吸引更廣泛的客戶群。
* **產業領先地位：**持續的創新幫助公司保持其在業界的領先地位，在競爭日益激烈的環境中脫穎而出。

**股價的持續上漲傾向**

公司的創新策略在投資者中引起了極大的興趣，並導致其股價出現持續上漲趨勢。股價反映了投資者對以下方面的預期：

* **亮眼的長期增長：**持續的創新被視為公司未來增長的關鍵驅動力。
* **穩健的財務業績：**創新策略已導致收入和利潤的強勁增長，提振了投資者的信心。
* **競爭優勢：**公司對創新的承諾使其在同業中獲得了顯著的競爭優勢，這是投資者看重的。
* **市場滲透：**預計公司的創新會擴大其市場滲透率，為未來的增長創造新的機會。

總體而言，公司的創新策略是其成功和股價持續上漲的基礎。通過持續投資於研發、採用開放創新方法並專注於客戶導向，公司已定位自己以在未來創造顯著的長期增長。
__________________________________________________
